In [1]:
import pandas as pd 
import numpy as np 

In [57]:
from sklearn.model_selection import KFold, cross_val_score , cross_validate
from sklearn.linear_model import LinearRegression ,Ridge,Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [6]:
df = pd.read_csv(r"C:\Users\avanindra Bose\OneDrive\Desktop\Real State Project\Cleaned Datasets\gurgaon_properties_post_feature_selection_v2.csv")

In [7]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,0.0,Low,Low Floor
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,0.0,Low,Mid Floor
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,0.0,Low,High Floor
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,1.0,High,Mid Floor
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,0.0,High,Mid Floor


In [8]:
df['furnishing_type'].value_counts()

furnishing_type
0.0    2349
1.0    1018
2.0     187
Name: count, dtype: int64

In [9]:
df['furnishing_type'] = df['furnishing_type'].replace({0.0:'unfurnished',1.0:'semifurnished',2.0:'furnished'})

In [32]:
df.sample(5)

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
743,house,sector 105,1.15,3.0,2.0,1,Moderately Old,3889.0,0.0,0.0,unfurnished,Low,Low Floor
913,flat,sector 92,0.95,4.0,3.0,2,New Property,1983.0,0.0,0.0,unfurnished,Low,High Floor
216,flat,sector 81,2.29,3.0,4.0,3+,Relatively New,1900.0,1.0,0.0,furnished,Medium,High Floor
3393,flat,sector 72,3.30,3.0,4.0,3+,New Property,2185.0,1.0,0.0,furnished,High,High Floor
1520,flat,sector 37d,1.08,3.0,3.0,3,Relatively New,1700.0,0.0,0.0,unfurnished,High,Mid Floor


In [11]:
X = df.drop(columns=['price'])
y = df['price']

In [12]:
y_transformed = np.log1p(y)

# Ordinal Encoding

In [33]:
num_col = ['bedRoom','bathroom','built_up_area','servant room' , 'store room']

In [34]:
cat_col = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [35]:
preprocess = ColumnTransformer(
    transformers= [
        ('num',StandardScaler(), num_col),
        ('cat' ,OrdinalEncoder() , cat_col)
    ] , 
    remainder="passthrough"
)

In [36]:
pipeline = Pipeline(
    [
        ('preprocess' , preprocess),
        ('estimator' , LinearRegression())
    ]
)

In [37]:
k_fold = KFold(n_splits=10 , shuffle=True , random_state=42)
scores = cross_val_score(pipeline,X,y_transformed,cv=k_fold , scoring='r2')

In [39]:
scores.mean()

np.float64(0.7363096633436828)

In [41]:
scores.std()

np.float64(0.03238005754429936)

In [42]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [43]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocess', ...), ('estimator', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [44]:
y_pred = pipeline.predict(X_test)

In [45]:
y_pred = np.expm1(y_pred)

In [46]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.9463822160089356

In [49]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocess),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [50]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor(),
    'KNN' : KNeighborsRegressor()
}

In [51]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [52]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [53]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.889488,0.504048
5,random forest,0.880289,0.536237
6,extra trees,0.868155,0.548131
7,gradient boosting,0.872611,0.575952
11,KNN,0.803658,0.668036
9,mlp,0.813489,0.714517
4,decision tree,0.769772,0.733718
1,svr,0.764201,0.847264
8,adaboost,0.763069,0.850916
2,ridge,0.736313,0.946339


In [58]:
scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error"
}

In [59]:
k_fold = KFold(n_splits=10 , shuffle=True , random_state=42)
scores = cross_validate(pipeline,X,y_transformed,cv=k_fold , scoring=scoring)

In [61]:
print("Mean R2:", scores["test_r2"].mean())
print("Mean MAE:", -scores["test_mae"].mean())

Mean R2: 0.7363096633436828
Mean MAE: 0.21293592367164615


# One Hot Encoding

In [62]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_col),
        ('cat', OrdinalEncoder(), cat_col),
        ('cat1',OneHotEncoder(drop='first'),['sector','agePossession','furnishing_type'])
    ], 
    remainder='passthrough'
)

In [63]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [64]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [65]:
scores.mean()

np.float64(0.8546094810971422)

In [66]:
scores.std()

np.float64(0.015997422908695623)

In [67]:

X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [68]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [69]:
y_pred = pipeline.predict(X_test)

In [70]:

y_pred = np.expm1(y_pred)

In [71]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.6497514315131458

In [72]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [73]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [74]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

c:\Users\avanindra Bose\OneDrive\Desktop\Real State Project\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [75]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [76]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.893706,0.460849
10,xgboost,0.895850,0.493456
5,random forest,0.890726,0.500973
7,gradient boosting,0.876369,0.566923
9,mlp,0.871748,0.608029
0,linear_reg,0.854609,0.649751
2,ridge,0.854739,0.652915
4,decision tree,0.806764,0.700962
1,svr,0.769741,0.834124
8,adaboost,0.754739,0.837752


# Target Encoder

In [80]:
import category_encoders as ce

columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [81]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [82]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [83]:
scores.mean(),scores.std()

(np.float64(0.8295219182255362), np.float64(0.018384463379122782))

In [84]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [85]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [86]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [87]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [88]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.904798,0.447518
6,extra trees,0.902091,0.452926
5,random forest,0.900611,0.460506
7,gradient boosting,0.888958,0.508507
4,decision tree,0.826431,0.547384
9,mlp,0.854207,0.583055
8,adaboost,0.815677,0.697422
0,linear_reg,0.829522,0.713011
2,ridge,0.829536,0.713523
1,svr,0.782917,0.818851


# Hence We can say that the best model for Our Dataset was XGBoost and Random Forest. 

# HyperParameter Tuning Using Optuna

In [94]:
import optuna

c:\Users\avanindra Bose\OneDrive\Desktop\Real State Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [179]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [180]:
columns_to_encode = ['property_type','balcony', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [181]:
def mae_original(y_true, y_pred):
    return mean_absolute_error(np.expm1(y_true), np.expm1(y_pred))

mae_scorer = make_scorer(mae_original, greater_is_better=False)

In [197]:
def objective(trial):
    params = {
        
    "objective": "reg:squarederror",
    "booster": "gbtree",  # safest choice for tabular
    "n_estimators": trial.suggest_int("n_estimators", 100, 5000),  # more trees = safer, especially with small learning_rate
    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.5, log=True),  # controls step size
    "max_depth": trial.suggest_int("max_depth", 3, 12),  # tree complexity
    "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),  # min samples per leaf
    "subsample": trial.suggest_float("subsample", 0.7, 1.0),  # row sampling
    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),  # feature sampling
    "gamma": trial.suggest_float("gamma", 0, 5),  # split penalty
    "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 3.0, log=True),  # L1 regularization
    "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 3.0, log=True),  # L2 regularizatio

    }


    model = Pipeline([
        ('preprocessor', preprocessor),   
        ('regressor', XGBRegressor(**params, random_state=42, n_jobs=-1))
    ])

    model.fit(X_train,y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(np.expm1(y_test) , np.expm1(preds))
    # r2 = r2_score(np.expm1(y_test),preds)

    return mae 


In [205]:
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=300)  # Run 50 trials to find the best hyperparameters


[I 2025-09-05 01:23:56,954] A new study created in memory with name: no-name-7f8adad3-cba3-4f28-a197-7abd7060e84d
[I 2025-09-05 01:23:57,523] Trial 0 finished with value: 0.6133130060603683 and parameters: {'n_estimators': 3245, 'learning_rate': 0.16187286662927378, 'max_depth': 12, 'min_child_weight': 3, 'subsample': 0.9947802144649395, 'colsample_bytree': 0.9272626679215805, 'gamma': 3.4745574180926075, 'reg_alpha': 0.0013207744614442398, 'reg_lambda': 0.004904108250909164}. Best is trial 0 with value: 0.6133130060603683.
[I 2025-09-05 01:23:58,018] Trial 1 finished with value: 0.5981178457257449 and parameters: {'n_estimators': 2563, 'learning_rate': 0.3157165840669792, 'max_depth': 5, 'min_child_weight': 4, 'subsample': 0.8661740251529855, 'colsample_bytree': 0.8236910478024894, 'gamma': 2.7475512350550395, 'reg_alpha': 1.8798702536997316e-06, 'reg_lambda': 0.00028564247996054647}. Best is trial 1 with value: 0.5981178457257449.
[I 2025-09-05 01:23:58,211] Trial 2 finished with val

In [199]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.4327019562835264
Best hyperparameters: {'n_estimators': 2247, 'learning_rate': 0.029607581182756134, 'max_depth': 8, 'min_child_weight': 1, 'subsample': 0.8776300145819186, 'colsample_bytree': 0.7140091435602947, 'gamma': 0.0014673093687880855, 'reg_alpha': 7.817104638830973e-06, 'reg_lambda': 1.1448017199516563e-07}


In [206]:
best_params = {
    "n_estimators": 2247,
    "learning_rate": 0.029607581182756134,
    "max_depth": 8,
    "min_child_weight": 1,
    "subsample": 0.8776300145819186,
    "colsample_bytree": 0.7140091435602947,
    "gamma": 0.0014673093687880855,
    "reg_alpha": 7.817104638830973e-06,
    "reg_lambda": 1.1448017199516563e-07,
    "random_state": 42,
    "n_jobs": -1
}

In [207]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Split hold-out test set
X_trainval, X_final_test, y_trainval, y_final_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)

# Fit on training+validation
final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(**best_params))
])
final_model.fit(X_trainval, y_trainval)

# Predict on final test
y_pred_final = final_model.predict(X_final_test)

print("Hold-out R²:", r2_score(y_final_test, y_pred_final))
print("Hold-out MAE:", mean_absolute_error(np.expm1(y_final_test), np.expm1(y_pred_final)))


Hold-out R²: 0.920148633377009
Hold-out MAE: 0.4327019562835264


In [ ]:
def objective_rf(trial):
    params = {
        
    # number of trees (higher → stabler, but slower)
    "n_estimators": trial.suggest_int("n_estimators", 200, 5000, step=100),

    # maximum depth of trees (None = full growth, but usually overfits)
    "max_depth": trial.suggest_int("max_depth", 5, 30),

    # minimum samples required to split a node
    "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),

    # minimum samples required at a leaf
    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),

    # number of features to consider at each split
    "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),

    # whether to use bootstrap sampling
    "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
}



    model = Pipeline([
        ('preprocessor', preprocessor),   
        ('regressor', RandomForestRegressor(**params, random_state=42, n_jobs=-1))
    ])

    model.fit(X_train,y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(np.expm1(y_test) , np.expm1(preds))
    # r2 = r2_score(np.expm1(y_test),preds)

    return mae 

In [ ]:
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective_rf, n_trials=300)

# Exporting the Model

In [208]:
columns_to_encode = ['property_type','balcony', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [209]:
final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(**best_params))
])

In [210]:
final_model.fit(X,y_transformed)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [211]:
import pickle

with open("final_xgb_pipeline.pkl", "wb") as f:
    pickle.dump(final_model, f)


In [212]:
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)

In [213]:
X

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,3.0,2.0,2,New Property,850.0,0.0,0.0,unfurnished,Low,Low Floor
1,flat,sector 89,2.0,2.0,2,New Property,1226.0,1.0,0.0,unfurnished,Low,Mid Floor
2,flat,sohna road,2.0,2.0,1,New Property,1000.0,0.0,0.0,unfurnished,Low,High Floor
3,flat,sector 92,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,semifurnished,High,Mid Floor
4,flat,sector 102,2.0,2.0,1,Relatively New,582.0,0.0,1.0,unfurnished,High,Mid Floor
...,...,...,...,...,...,...,...,...,...,...,...,...
3549,flat,sector 84,2.0,2.0,1,Relatively New,532.0,0.0,0.0,unfurnished,Medium,Mid Floor
3550,house,sector 109,5.0,5.0,3+,Relatively New,6228.0,1.0,1.0,unfurnished,High,Low Floor
3551,flat,sector 2,1.0,1.0,1,Moderately Old,665.0,0.0,0.0,semifurnished,Medium,Mid Floor
3552,house,sector 43,5.0,6.0,3,Moderately Old,5490.0,1.0,1.0,unfurnished,Medium,Mid Floor


# Testing out the Model

In [214]:
X.columns

Index(['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category'],
      dtype='object')

In [215]:
X.iloc[0].values

array(['flat', 'sector 36', np.float64(3.0), np.float64(2.0), '2',
       'New Property', np.float64(850.0), np.float64(0.0),
       np.float64(0.0), 'unfurnished', 'Low', 'Low Floor'], dtype=object)

In [219]:
data = [['house', 'sector 81', 2, 3, '3+', 'New Property', 1345, 0, 0, 'unfurnished', 'Low', 'Low Floor']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category']

one_df = pd.DataFrame(data, columns=columns)

one_df

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 81,2,3,3+,New Property,1345,0,0,unfurnished,Low,Low Floor


In [220]:
np.expm1(final_model.predict(one_df))

array([1.3568053], dtype=float32)

In [221]:
X.dtypes

property_type       object
sector              object
bedRoom            float64
bathroom           float64
balcony             object
agePossession       object
built_up_area      float64
servant room       float64
store room         float64
furnishing_type     object
luxury_category     object
floor_category      object
dtype: object